In [45]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

🔵 TRAIN AND TEST LASSO REGRESSION ON B2 MICROGLOBULIN

In [46]:
clinical_df = pd.read_csv(r"D:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")
numeric_clinical = clinical_df.select_dtypes(include=['number']).columns

features_df = pd.read_csv(r"D:\CSV\merged\merged_CaSupp_25_radiomics_spine_lesions_features.csv")
numeric_features = features_df.select_dtypes(include=['number']).columns

scaler = StandardScaler()
clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df[numeric_clinical]),
                               columns=clinical_df[numeric_clinical].columns,
                               index=clinical_df[numeric_clinical].index)

features_scaled = pd.DataFrame(scaler.fit_transform(features_df[numeric_features]),
                               columns=features_df[numeric_features].columns,
                               index=features_df[numeric_features].index)

valid_idx = clinical_scaled['Beta2 microglobulin (mg/l)'].notna()
lasso = Lasso(alpha=0.01)

X = features_scaled[numeric_features][valid_idx]
y = clinical_scaled['Beta2 microglobulin (mg/l)'][valid_idx]

lasso.fit(X, y)
print("R^2 score Lasso:", lasso.score(X, y))
print("Feature weights:")
feature_weights = pd.Series(lasso.coef_, index=X.columns)
feature_weights = feature_weights.sort_values(ascending=False)
feature_weights

R^2 score Lasso: 0.7719541089448922
Feature weights:


original_shape_MajorAxisLength                        0.939938
original_shape_SurfaceVolumeRatio                     0.448924
gradient_gldm_GrayLevelNonUniformity                  0.405301
gradient_gldm_LargeDependenceHighGrayLevelEmphasis    0.284570
original_glcm_MCC                                     0.248328
                                                        ...   
original_glszm_ZonePercentage                        -0.449097
original_shape_Maximum3DDiameter                     -0.461047
n_lesions                                            -0.478532
gradient_ngtdm_Coarseness                            -0.562481
gradient_glcm_Imc1                                   -0.766993
Length: 201, dtype: float64

🔵 LASSO FOR EVERY DATASET

In [47]:
clinical_df = pd.read_csv(r"D:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")
numeric_clinical = clinical_df.select_dtypes(include=['number']).columns

df_c25 = pd.read_csv(r"D:\CSV\merged\merged_CaSupp_25_radiomics_spine_lesions_features.csv")
df_vmi40 = pd.read_csv(r"D:\CSV\merged\merged_monoe_40kev_radiomics_spine_lesions_features.csv")
df_vmi80 = pd.read_csv(r"D:\CSV\merged\merged_monoe_80kev_radiomics_spine_lesions_features.csv")
df_vmi120 = pd.read_csv(r"D:\CSV\merged\merged_monoe_120kev_radiomics_spine_lesions_features.csv")
df_konv = pd.read_csv(r"D:\CSV\merged\merged_konv_radiomics_spine_lesions_features.csv")

dfs = [df_c25, df_konv, df_vmi40, df_vmi80, df_vmi120]
dataset_names = ['CaSupp_25', 'Konv', 'VMI_40', 'VMI_80', 'VMI_120']

for df, name in zip(dfs, dataset_names):
    numeric_features = df.select_dtypes(include=['number']).columns

    scaler = StandardScaler()
    clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df[numeric_clinical]),
                                   columns=clinical_df[numeric_clinical].columns,
                                   index=clinical_df[numeric_clinical].index)

    features_scaled = pd.DataFrame(scaler.fit_transform(features_df[numeric_features]),
                                   columns=features_df[numeric_features].columns,
                                   index=features_df[numeric_features].index)

    valid_idx = clinical_scaled['Beta2 microglobulin (mg/l)'].notna()
    lasso = Lasso(alpha=0.01)

    X = features_scaled[numeric_features][valid_idx]
    y = clinical_scaled['Beta2 microglobulin (mg/l)'][valid_idx]

    lasso.fit(X, y)
    print(f"Dataset: {name}")
    print("R^2 score Lasso:", lasso.score(X, y))
    print("Feature weights:")
    feature_weights = pd.Series(lasso.coef_, index=X.columns)
    feature_weights = feature_weights.sort_values(ascending=False)
    print(feature_weights)
    print()

Dataset: CaSupp_25
R^2 score Lasso: 0.7719541089448922
Feature weights:
original_shape_MajorAxisLength                        0.939938
original_shape_SurfaceVolumeRatio                     0.448924
gradient_gldm_GrayLevelNonUniformity                  0.405301
gradient_gldm_LargeDependenceHighGrayLevelEmphasis    0.284570
original_glcm_MCC                                     0.248328
                                                        ...   
original_glszm_ZonePercentage                        -0.449097
original_shape_Maximum3DDiameter                     -0.461047
n_lesions                                            -0.478532
gradient_ngtdm_Coarseness                            -0.562481
gradient_glcm_Imc1                                   -0.766993
Length: 201, dtype: float64

Dataset: Konv
R^2 score Lasso: 0.7719541089448922
Feature weights:
original_shape_MajorAxisLength                        0.939938
original_shape_SurfaceVolumeRatio                     0.448924
gradient_gldm

🔵 LASSO FOR EVERY DATASET AND JUST SIGNIFICANT FEATURES

In [48]:
clinical_df = pd.read_csv(r"D:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")
numeric_clinical = clinical_df.select_dtypes(include=['number']).columns

df_c25 = pd.read_csv(r"D:\CSV\merged\merged_CaSupp_25_radiomics_spine_lesions_features.csv")
df_vmi40 = pd.read_csv(r"D:\CSV\merged\merged_monoe_40kev_radiomics_spine_lesions_features.csv")
df_vmi80 = pd.read_csv(r"D:\CSV\merged\merged_monoe_80kev_radiomics_spine_lesions_features.csv")
df_vmi120 = pd.read_csv(r"D:\CSV\merged\merged_monoe_120kev_radiomics_spine_lesions_features.csv")
df_konv = pd.read_csv(r"D:\CSV\merged\merged_konv_radiomics_spine_lesions_features.csv")

sp_c25 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_CaSupp_25.csv")
sp_konv = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_konv.csv")
sp_vmi40 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_vmi40.csv")
sp_vmi80 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_vmi80.csv")
sp_vmi120 = pd.read_csv(r"D:\CSV\spearman_coef\spearman_results_vmi120.csv")

significant_c25 = sp_c25[sp_c25["p_value"] < 0.05]['Feature']
significant_konv = sp_konv[sp_konv["p_value"] < 0.05]['Feature']
significant_vmi40 = sp_vmi40[sp_vmi40["p_value"] < 0.05]['Feature']
significant_vmi80 = sp_vmi80[sp_vmi80["p_value"] < 0.05]['Feature']
significant_vmi120 = sp_vmi120[sp_vmi120["p_value"] < 0.05]['Feature']

dfs = [df_c25, df_konv, df_vmi40, df_vmi80, df_vmi120]
dataset_names = ['CaSupp_25', 'Konv', 'VMI_40', 'VMI_80', 'VMI_120']
significant = [significant_c25, significant_konv, significant_vmi40, significant_vmi80, significant_vmi120]

for df, name, significant_features in zip(dfs, dataset_names, significant):

    scaler = StandardScaler()
    clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df[numeric_clinical]),
                                   columns=clinical_df[numeric_clinical].columns,
                                   index=clinical_df[numeric_clinical].index)

    features_scaled = pd.DataFrame(scaler.fit_transform(df[significant_features]),
                                   columns=df[significant_features].columns,
                                   index=df[significant_features].index)

    valid_idx = clinical_scaled['Beta2 microglobulin (mg/l)'].notna()
    lasso = Lasso(alpha=0.01)

    X = features_scaled[significant_features][valid_idx]
    y = clinical_scaled['Beta2 microglobulin (mg/l)'][valid_idx]

    lasso.fit(X, y)
    print(f"Dataset: {name}")
    print("R^2 score Lasso:", lasso.score(X, y))
    print("Feature weights:")
    feature_weights = pd.Series(lasso.coef_, index=X.columns)
    feature_weights = feature_weights.sort_values(ascending=False)
    print(feature_weights)
    print()

Dataset: CaSupp_25
R^2 score Lasso: 0.3457971911800738
Feature weights:
gradient_firstorder_90Percentile                   0.361174
gradient_gldm_DependenceVariance                   0.321074
original_firstorder_MeanAbsoluteDeviation          0.247478
original_glszm_GrayLevelNonUniformityNormalized    0.214876
original_firstorder_Kurtosis                       0.179818
                                                     ...   
gradient_glrlm_LowGrayLevelRunEmphasis            -0.276104
original_firstorder_Median                        -0.312193
original_glszm_ZoneEntropy                        -0.339129
gradient_glcm_Imc1                                -0.346759
gradient_firstorder_InterquartileRange            -0.565485
Length: 100, dtype: float64

Dataset: Konv
R^2 score Lasso: 0.422345455170131
Feature weights:
original_firstorder_InterquartileRange               0.706189
original_glcm_Contrast                               0.457280
original_firstorder_10Percentile                 